### Steps followed
1. Load resolved dataset and confirm baseline
2. Drop leakage columns
3. Drop high-missing and administrative columns
4. Drop zero-signal columns
5. Impute missing values
6. Apply transformations
7. Temporal train/test split
8. Save processed datasets
9. Cleaning summary — before vs after

In [72]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)

print(f"pandas:  {pd.__version__}")
print(f"numpy:   {np.__version__}")

pandas:  3.0.1
numpy:   2.4.3


In [73]:
df = pd.read_pickle("../data/processed/df_resolved.pkl")

print("df_resolved loaded successfully")
print(f"Shape:         {df.shape}")
print(f"Rows:          {len(df):,}")
print(f"Columns:       {df.shape[1]}")
print(f"Default rate:  {df['default_flag'].mean()*100:.2f}%")
print(f"Memory usage:  {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print()
print("dtypes summary:")
print(df.dtypes.value_counts())

df_resolved loaded successfully
Shape:         (1348099, 154)
Rows:          1,348,099
Columns:       154
Default rate:  19.98%
Memory usage:  3.78 GB

dtypes summary:
float64           113
str                38
int64               1
datetime64[us]      1
int32               1
Name: count, dtype: int64


## Section 1: Drop Post-Origination Leakage Columns

In [74]:
leakage_columns = [
    "out_prncp", "out_prncp_inv",
    "total_pymnt", "total_pymnt_inv",
    "total_rec_prncp", "total_rec_int", "total_rec_late_fee",
    "recoveries", "collection_recovery_fee",
    "last_pymnt_d", "last_pymnt_amnt", "next_pymnt_d",
    "last_credit_pull_d",
    "last_fico_range_high", "last_fico_range_low",
    "hardship_flag", "hardship_type", "hardship_reason", "hardship_status",
    "deferral_term", "hardship_amount", "hardship_start_date",
    "hardship_end_date", "hardship_length", "hardship_dpd",
    "hardship_loan_status", "hardship_payoff_balance_amount",
    "hardship_last_payment_amount", "payment_plan_start_date",
    "orig_projected_additional_accrued_interest",
    "settlement_amount", "settlement_percentage", "settlement_term",
    "settlement_status", "settlement_date", "debt_settlement_flag_date",
]

# Confirm all present before dropping
leakage_present = [c for c in leakage_columns if c in df.columns]
leakage_missing = [c for c in leakage_columns if c not in df.columns]

print(f"Leakage columns found:    {len(leakage_present)}")
print(f"Leakage columns not found: {len(leakage_missing)}")

df = df.drop(columns=leakage_present)
print(f"\nShape after leakage drop: {df.shape}")
print(f"Columns removed: {len(leakage_present)}")

Leakage columns found:    36
Leakage columns not found: 0

Shape after leakage drop: (1348099, 118)
Columns removed: 36


## Section 2: Drop High-Missing, Administrative and Redundant Columns

In [75]:
# Tier 1 — >50% missing (from Notebook 01 analysis)
# Recompute missing summary on current df to get Tier 1 columns
total = len(df)
missing_pct = (df.isnull().sum() / total * 100)

# Exclude mths_since_last_delinq — borderline at 50.4%, handled
# separately in Section 4 (ever_delinq indicator + sentinel impute)
tier1_cols = missing_pct[missing_pct > 50].index.tolist()
tier1_cols = [c for c in tier1_cols if c != "mths_since_last_delinq"]

# Administrative columns
admin_cols = [
    "id", "member_id", "url", "desc", "title",
    "zip_code", "policy_code"
]

# Redundant columns identified in Section 7 correlation analysis
redundant_cols = ["funded_amnt", "funded_amnt_inv"]

# Helper columns created in Notebook 01 — not features
helper_cols = ["issue_d_parsed"]

# Combine all, filter to those present in df
all_drop = list(set(tier1_cols + admin_cols + redundant_cols + helper_cols))
all_drop_present = [c for c in all_drop if c in df.columns]

print(f"Tier 1 (>50% missing):  {len([c for c in tier1_cols if c in df.columns])}")
print(f"Administrative:         {len([c for c in admin_cols if c in df.columns])}")
print(f"Redundant:              {len([c for c in redundant_cols if c in df.columns])}")
print(f"Helper:                 {len([c for c in helper_cols if c in df.columns])}")
print(f"Total to drop:          {len(all_drop_present)}")

df = df.drop(columns=all_drop_present)

print(f"\nShape after drop: {df.shape}")

Tier 1 (>50% missing):  36
Administrative:         7
Redundant:              2
Helper:                 1
Total to drop:          44

Shape after drop: (1348099, 74)


# Section 3: Drop Zero-Signal Columns

In [76]:
# Confirmed no discriminatory power in Section 5 of Notebook 01
zero_signal_cols = [
    "initial_list_status",
    "disbursement_method",
    "pymnt_plan",
    "application_type",
    "emp_title",       # free text, high cardinality
    "addr_state",      # narrow default rate range across states
]

zero_signal_present = [c for c in zero_signal_cols if c in df.columns]
zero_signal_missing = [c for c in zero_signal_cols if c not in df.columns]

print(f"Zero-signal columns found:     {len(zero_signal_present)}")
print(f"Zero-signal columns not found: {len(zero_signal_missing)}")

if zero_signal_missing:
    print(f"  Not found: {zero_signal_missing}")

df = df.drop(columns=zero_signal_present)

print(f"\nShape after zero-signal column drop: {df.shape}")

Zero-signal columns found:     6
Zero-signal columns not found: 0

Shape after zero-signal column drop: (1348099, 68)


## Section 4: Impute Missing Values

In [77]:
# Check which of our target imputation columns are still present
target_cols = [
    "mths_since_last_delinq",
    "mths_since_recent_inq",
    "emp_length",
    "revol_util"
]

for col in target_cols:
    present = col in df.columns
    print(f"{col:<35} present: {present}")

mths_since_last_delinq              present: True
mths_since_recent_inq               present: True
emp_length                          present: True
revol_util                          present: True


In [78]:
# mths_since_last_delinq — missing means never delinquent
# Create binary indicator before imputing
df["ever_delinq"] = df["mths_since_last_delinq"].notnull().astype(int)
df["mths_since_last_delinq"] = df["mths_since_last_delinq"].fillna(999)

print(f"ever_delinq created:")
print(f"  Ever delinquent:     {df['ever_delinq'].sum():>10,}")
print(f"  Never delinquent:    {(df['ever_delinq']==0).sum():>10,}")

# mths_since_recent_inq — missing means no recent inquiry
df["mths_since_recent_inq"] = df["mths_since_recent_inq"].fillna(999)
print(f"\nmths_since_recent_inq imputed with 999")

# emp_length — impute missing with Unknown category
df["emp_length"] = df["emp_length"].fillna("Unknown")
print(f"emp_length imputed with 'Unknown'")

# revol_util — impute with median
revol_util_median = df["revol_util"].median()
df["revol_util"] = df["revol_util"].fillna(revol_util_median)
print(f"revol_util imputed with median: {revol_util_median:.2f}")

# Remaining numeric columns — median impute
numeric_cols_remaining = df.select_dtypes(include="number").columns.tolist()
exclude_from_impute = ["default_flag", "issue_year", "ever_delinq"]
numeric_cols_remaining = [c for c in numeric_cols_remaining
                          if c not in exclude_from_impute]

imputed = []
for col in numeric_cols_remaining:
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        imputed.append((col, n_missing, median_val))

print(f"\nRemaining numeric columns imputed with median: {len(imputed)}")
for col, n, val in imputed:
    print(f"  {col:<35} missing: {n:>7,}   median: {val:.4f}")

# Remaining categorical columns — mode impute
cat_cols = df.select_dtypes(include="str").columns.tolist()
cat_imputed = []
for col in cat_cols:
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        cat_imputed.append((col, n_missing, mode_val))

print(f"\nCategorical columns imputed with mode: {len(cat_imputed)}")
for col, n, val in cat_imputed:
    print(f"  {col:<35} missing: {n:>7,}   mode: {val}")

# Confirm no missing values remain
total_missing = df.isnull().sum().sum()
print(f"\nTotal missing values remaining: {total_missing}")

ever_delinq created:
  Ever delinquent:        668,139
  Never delinquent:       679,960

mths_since_recent_inq imputed with 999
emp_length imputed with 'Unknown'
revol_util imputed with median: 52.20

Remaining numeric columns imputed with median: 46
  annual_inc                          missing:       4   median: 65000.0000
  dti                                 missing:     374   median: 17.6100
  delinq_2yrs                         missing:      29   median: 0.0000
  inq_last_6mths                      missing:      30   median: 0.0000
  open_acc                            missing:      29   median: 11.0000
  pub_rec                             missing:      29   median: 0.0000
  total_acc                           missing:      29   median: 23.0000
  collections_12_mths_ex_med          missing:     145   median: 0.0000
  acc_now_delinq                      missing:      29   median: 0.0000
  tot_coll_amt                        missing:  70,276   median: 0.0000
  tot_cur_bal        

In [79]:
# Cap impossible DTI values — 8 rows with dti > 100 (max 999.0)
# These are data entry errors, not genuine observations
# 99th percentile is 37.3 — cap at 100 as conservative upper bound

dti_cap = 100
n_capped = (df["dti"] > dti_cap).sum()
df["dti"] = df["dti"].clip(upper=dti_cap)

print(f"dti capped at {dti_cap}")
print(f"  Rows affected: {n_capped}")
print(f"  New max:       {df['dti'].max():.2f}")

dti capped at 100
  Rows affected: 533
  New max:       100.00


In [80]:
# Merge rare home_ownership categories into OTHER
# ANY (3 obs) and NONE (50 obs) are too sparse for reliable WoE estimation

rare_map = {"ANY": "OTHER", "NONE": "OTHER"}
df["home_ownership"] = df["home_ownership"].replace(rare_map)

print("home_ownership after merging ANY and NONE into OTHER:")
print(df["home_ownership"].value_counts())

home_ownership after merging ANY and NONE into OTHER:
home_ownership
MORTGAGE    666852
RENT        535699
OWN         145027
OTHER          521
Name: count, dtype: int64


## Section 5: Transformations

In [81]:
# log_annual_inc — annual_inc is heavily right-skewed
df["log_annual_inc"] = np.log1p(df["annual_inc"])

# Drop original annual_inc — log version replaces it
df = df.drop(columns=["annual_inc"])

print("log_annual_inc created (log1p of annual_inc)")
print(f"  Mean log_annual_inc: {df['log_annual_inc'].mean():.4f}")
print(f"  Std log_annual_inc:  {df['log_annual_inc'].std():.4f}")
print()

# Drop issue_d — raw date string, issue_year already extracted
if "issue_d" in df.columns:
    df = df.drop(columns=["issue_d"])
    
print(f"\nShape after transformations: {df.shape}")

log_annual_inc created (log1p of annual_inc)
  Mean log_annual_inc: 11.0816
  Std log_annual_inc:  0.5707


Shape after transformations: (1348099, 68)


## Section 6: Temporal Train/Test Split 

In [82]:
df_train = df[df["issue_year"] <= 2015].copy()
df_test  = df[df["issue_year"] >= 2016].copy()

print(f"Train set: {df_train.shape}  ({df_train['issue_year'].min()}–{df_train['issue_year'].max()})")
print(f"Test set:  {df_test.shape}  ({df_test['issue_year'].min()}–{df_test['issue_year'].max()})")
print()
print(f"Train default rate: {df_train['default_flag'].mean()*100:.2f}%")
print(f"Test default rate:  {df_test['default_flag'].mean()*100:.2f}%")
print()
print(f"Train share of total: {len(df_train)/len(df)*100:.1f}%")
print(f"Test share of total:  {len(df_test)/len(df)*100:.1f}%")

Train set: (829355, 68)  (2007–2015)
Test set:  (518744, 68)  (2016–2018)

Train default rate: 18.46%
Test default rate:  22.42%

Train share of total: 61.5%
Test share of total:  38.5%


## Section 7: Save Processed Datasets 


In [83]:
os.makedirs("../data/processed", exist_ok=True)

df.to_pickle("../data/processed/df_clean.pkl")
df_train.to_pickle("../data/processed/df_train.pkl")
df_test.to_pickle("../data/processed/df_test.pkl")

# Verify all three
for name, path in [
    ("df_clean", "../data/processed/df_clean.pkl"),
    ("df_train", "../data/processed/df_train.pkl"),
    ("df_test",  "../data/processed/df_test.pkl"),
]:
    size_mb = os.path.getsize(path) / 1e6
    df_check = pd.read_pickle(path)
    print(f"{name:<12} shape: {str(df_check.shape):<20} size: {size_mb:.1f} MB")

df_clean     shape: (1348099, 68)        size: 666.0 MB
df_train     shape: (829355, 68)         size: 409.7 MB
df_test      shape: (518744, 68)         size: 256.2 MB


#### Column Reduction

| Action | Section | Columns Removed |
|--------|---------|----------------|
| Leakage columns dropped | 1 | 36 |
| Tier 1 >50% missing dropped | 2 | 37 |
| Administrative columns dropped | 2 | 7 |
| Redundant columns dropped | 2 | 2 |
| Helper columns dropped | 2 | 1 |
| Zero-signal columns dropped | 3 | 6 |
| Transformation drops (annual_inc, issue_d) | 5 | 2 |
| **Total removed** | | **91** |




| Action | Section | Columns Added |
|--------|---------|--------------|
| ever_delinq (binary delinquency indicator) | 4 | 1 |
| log_annual_inc (log transform of annual_inc) | 5 | 1 |
| **Total added** | | **2** |

#### Imputation Summary

- `mths_since_last_delinq` — sentinel value 999 + `ever_delinq` binary flag
- `mths_since_recent_inq` — sentinel value 999
- `emp_length` — "Unknown" category
- `revol_util` — median (52.20)
- 46 remaining numeric columns — median impute
- `earliest_cr_line` — mode impute
- **Total missing values remaining: 0**


#### Final Datasets

| Dataset | Rows | Columns | Default Rate | Size |
|---------|------|---------|-------------|------|
| df_clean | 1,348,099 | 68 | 19.98% | 666.0 MB |
| df_train | 829,355 | 68 | 18.46% | 409.7 MB |
| df_test | 518,744 | 68 | 22.46% | 256.2 MB |

**Temporal split:** train 2007–2015, test 2016–2018. No random split.
The higher test default rate (22.42% vs 18.46%) reflects the rising
default trend observed in 2016–2017 and mirrors real deployment conditions.